# Sprint 6 — First Real Training Run

## Objective

Every sprint so far verified mechanics on tiny synthetic slices (Sprint 5's
32-image overfit check ran entirely on CPU). This sprint's goal was
narrower and more concrete: get from "the pipeline doesn't crash" to "the
model is actually training, on the real GPU, with the infrastructure
(experiment tracking, checkpoint/resume, mixed precision) that a long
full-dataset run will depend on" — and prove all of it together, not each
piece in isolation.

This notebook documents what changed, why, and what the first real run
produced. Each section follows cause → change → effect.

## 1. The GPU Was Never Actually Being Used

**Cause:** every earlier sprint (3, 5) installed PyTorch without ever
checking for a GPU — `pip install torch torchvision` defaulted to a
CPU-only build, and training "worked," so nobody questioned it.

**Discovery:** `nvidia-smi` (run directly from
`C:\Windows\System32\nvidia-smi.exe`, since it wasn't on `PATH`) showed an
NVIDIA GeForce RTX 3050 Laptop GPU — 4GB VRAM, driver 581.32, CUDA 13.0
supported — sitting completely idle.

**Fix:** uninstalled `torch`/`torchvision` (the `+cpu` build) and
reinstalled from PyTorch's `cu124` wheel index. `torch.cuda.is_available()`
now returns `True` and `torch.cuda.get_device_name(0)` reports the RTX
3050.

**Effect:** this is the single biggest change in this sprint — everything
below (mixed precision, realistic epoch counts) only makes sense once
training actually runs on the GPU instead of the CPU.

## 2. Experiment Tracking: MLflow + TensorBoard

**Why:** up to now, training metrics only ever existed as console output —
gone the moment the terminal closed, and impossible to compare across runs.
MLflow logs per-run parameters + metrics to a queryable store; TensorBoard
gives a live-updating loss/accuracy chart while training is running.

**Bug found:** MLflow's default tracking URI is built from the current
working directory's *absolute* path. That path contains a non-ASCII
character (the "ü" in the Windows username), which MLflow's URI encoding
mishandled — it produced a literal, non-existent folder name
(`S%C3%BCmeyye` instead of `Sümeyye`) and crashed with a `PermissionError`
trying to create it under `C:\Users\`.

**Fix:** `src/training/tracking.py` now explicitly sets
`mlflow.set_tracking_uri("sqlite:///mlflow.db")` — a relative path sidesteps
the encoding bug entirely, and a sqlite backend is required anyway since
MLflow 3.x deprecated the plain-filesystem store.

**Side effect noticed along the way:** installing `mlflow` pulled in a
`pandas` downgrade (3.0.3 → 2.3.3, one of its dependency constraints),
which surfaced a `groupby.apply` deprecation warning in the new
`subsample_per_class()` helper (section 6). Fixed by replacing `.apply()`
with a plain list comprehension + `pd.concat()` — same result, no
deprecated API.

## 3. Mixed Precision (AMP)

**Why:** the RTX 3050 has only 4GB of VRAM — tight for a ResNet-50 at
batch size 32+. Mixed precision (fp16 autocast for the forward/backward
pass, with a `GradScaler` to prevent gradient underflow) roughly halves
activation memory and runs faster on GPUs with tensor cores, which the
3050 has.

**What changed:** `train_one_epoch()`/`evaluate()` in `src/training/engine.py`
gained a `scaler`/`use_amp` parameter, wrapping the forward pass in
`torch.autocast(device_type=..., enabled=use_amp)` and routing the backward
pass through the scaler when enabled. Config gained
`training.mixed_precision: true`.

**Verified (in isolation, before the real run):** a 16-image batch used
about 400MB of VRAM under AMP — comfortable headroom on a 4GB card.

## 4. Checkpoint Resume

**Why:** before ever attempting a long run (the full 689K-image mushroom
set will take hours), we needed proof that interrupting training
(`Ctrl+C`) and resuming from the last checkpoint doesn't lose progress or
silently corrupt training state.

**What changed:** `save_checkpoint()` (`src/models/checkpoint.py`) now
optionally stores `epoch`, `optimizer_state_dict`, and `scaler_state_dict`
alongside the model weights. `scripts/train_baseline.py` gained a
`--resume <path>` flag that reloads all three and continues from
`epoch + 1` instead of epoch 0.

**Cause → effect if this had been skipped:** without saving optimizer
state, resuming would reset AdamW's per-parameter momentum/variance
estimates to zero — training would still "work" but show a visible loss
spike right after every resume, indistinguishable from an actual bug
without knowing to look for it.

## 5. Config Changes for a Real (Non-Smoke-Test) Run

Sprint 5's configs were tuned for a 1-epoch mechanical check, not an actual
training signal. Both `configs/mushroom.yaml` and `configs/flower.yaml`
changed identically:

| Key | Before (Sprint 5) | Now (Sprint 6) | Why |
|---|---|---|---|
| `batch_size` | 32 | 16 | headroom on the RTX 3050's 4GB VRAM |
| `pin_memory` | `false` | `true` | was a no-op on CPU; now speeds up the CPU→GPU transfer |
| `training.epochs` | 1 | 6 | a real (if still small) training budget instead of a single mechanical pass |
| `training.mixed_precision` | *(didn't exist)* | `true` | see section 3 |

## 6. Stratified Per-Class Subsetting

**Why:** the full mushroom dataset is 689,520 images — one real epoch at
this stage would take too long to use as a first correctness check, before
we even knew whether 6 epochs of real training would show a sensible loss
curve. A fast, representative slice was needed instead.

**What changed:** `src/data/sampling.py` gained `subsample_per_class()` —
takes up to N images *per class*, so every class is represented (unlike
naively truncating to the first N rows, which risks skipping whole species
outright given the mushroom CSV is grouped by species). `train_baseline.py`
gained a `--subset-per-class` flag wired to it.

**How much of each dataset this run actually used — the two datasets are
not comparable here:**

| | Mushroom | Flower |
|---|---|---|
| Full train set | 689,520 | 6,552 |
| Used this run (`--subset-per-class 25`) | 4,225 | 2,550 |
| **% of full train set used** | **0.61%** | **38.92%** |
| Full validation set | 15,616 | 818 |
| Used this run | **15,616 (100% — not subsetted)** | **818 (100% — not subsetted)** |

Two things worth being explicit about, since they change how much weight
these results should carry:

- **`--subset-per-class` only ever touches the training set.** Validation
  ran on the *full* validation set for both datasets — the val_accuracy
  numbers in sections 7–8 aren't diluted by subsetting, only the training
  data was.
- **25 images/class means something completely different for the two
  datasets.** For flower it's ~39% of all real training data — a
  legitimately large, representative slice. For mushroom it's 0.6% — a
  sliver. Any comparison between the two runs' results has to account for
  this before concluding anything about which dataset is "easier."

## 7-8. Results — Flower & Mushroom (moved)

Raw run results (hyperparameters, epoch-by-epoch tables, the MLflow/TensorBoard
pull code) now live in `notebooks/07_training_run_log.ipynb` under "Sprint 6", to
keep training-run logs out of this notebook. Section 8's reasoning below (Findings)
still refers to that data by number.


## 9. Findings

**The full system works end-to-end, not just its parts.** Sprint 5 verified
GPU-free mechanics (registry, freeze/unfreeze, checkpoint round-trip) on a
32-image overfit check. This sprint is the first time GPU training, mixed
precision, MLflow, TensorBoard, and checkpoint saving all ran together, on
a real (if still small) dataset slice, for both datasets — and produced
sensible, monotonically-improving training curves rather than just "didn't
crash."

**Flower converges fast and cleanly on a small stratified subset with a
pretrained backbone.** 92.3% validation accuracy after 6 epochs on ~25
images/class is consistent with Sprint 4's EfficientNet research note,
which reported 98.8% achievable on the *full* flower dataset — a small
subset getting into the low-90s quickly is the expected shape of that same
curve, not a surprise.

**Mushroom overfits much harder on the same-sized subset — but the primary
cause is the subset itself, not the model, pipeline, or config.** 25
images/class is 38.92% of flower's real training data but only 0.61% of
mushroom's (section 6) — mushroom was asked to solve a 169-class problem
from an artificially scarce slice of itself. The class-count difference
(169 vs. 102) and the `light` vs. `heavy` augmentation choice both make the
gap worse, but they're secondary, downstream effects of that same scarcity,
not independent causes (section 8 spells out why). **This result does not
mean mushroom's `augmentation_preset: light` decision was wrong** — that
decision assumed the real dataset's abundance (4,080 images/class), an
assumption this 25-images/class smoke test doesn't satisfy by construction.

**Neither result says anything about full-dataset performance yet.** Both
runs exist to prove the training infrastructure works, not to benchmark
either dataset — mushroom's overfitting here is a subset artifact, not
evidence about how it will train on its real 689,520 images.

**Both training curves are missing the tools to act on what they show —
but "add them" is the actionable takeaway, not "the config is broken."**
Flower's mild late-epoch val_loss uptick and mushroom's clear overfitting
both point at the same infrastructure gap: no LR scheduler, no early
stopping in `engine.py` yet (flagged already in Sprint 5's "Next Steps").

## 10. Next Steps

- Add LR scheduling and early stopping to `engine.py` before any longer
  run — this is infrastructure both datasets' curves are missing, not a
  reaction to mushroom's subset-driven overfitting specifically.
- Extrapolate full-dataset epoch time from the stratified-subset timing
  (~65s/epoch for ~2,550–4,225 images) to decide, together, whether full
  training runs locally on the RTX 3050 or moves to Kaggle — deliberately
  left as an open decision, not made automatically.
- **Do not change `configs/mushroom.yaml`'s `augmentation_preset` (or
  anything else) based on this smoke test.** If a larger-per-class subset
  run is used later to sanity-check the "artificial scarcity, not a config
  flaw" explanation from section 8, treat it as testing that hypothesis —
  not as a signal to preemptively "fix" a config that was set correctly
  for the real, 4,080-images/class dataset. The actual test is training on
  the real data.
- The Sprint 1 open questions (mushroom near-duplicate risk, flower's
  unlabeled test set) still block reporting a real, literature-comparable
  benchmark number — they don't block more training experiments, but they
  do block calling any number final.